**Library**

In [1]:
install.packages("fda")
install.packages("tvReg")
install.packages("readxl")
install.packages("tidyverse")
install.packages("dplyr")
install.packages("ggplot2")
install.packages("patchwork")
install.packages("writexl")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘mvtnorm’, ‘locfit’, ‘ash’, ‘FNN’, ‘kernlab’, ‘mclust’, ‘multicool’, ‘pracma’, ‘pcaPP’, ‘hdrcde’, ‘colorspace’, ‘ks’, ‘bitops’, ‘rainbow’, ‘RCurl’, ‘fds’, ‘deSolve’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘fracdiff’, ‘timeDate’, ‘cowplot’, ‘Deriv’, ‘forecast’, ‘microbenchmark’, ‘numDeriv’, ‘doBy’, ‘SparseM’, ‘MatrixModels’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘carData’, ‘abind’, ‘pbkrtest’, ‘quantreg’, ‘lme4’, ‘miscTools’, ‘rbibutils’, ‘car’, ‘lmtest’, ‘sandwich’, ‘strucchange’, ‘urca’, ‘RcppArmadillo’, ‘bdsmatrix’, ‘collapse’, ‘zoo’, ‘maxLik’, ‘Rdpack’, ‘Formula’, ‘systemfit’, ‘vars’, ‘bvarsv’, ‘plm’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package 

In [2]:
library(fda)
library(tvReg)
library(readxl)
library(fda)
library(dplyr)
library(tidyverse)
library(ggplot2)
library(patchwork)
library(writexl)

Loading required package: splines

Loading required package: fds

Loading required package: rainbow

Loading required package: MASS

Loading required package: pcaPP

Loading required package: RCurl

Loading required package: deSolve


Attaching package: ‘fda’


The following object is masked from ‘package:graphics’:

    matplot


The following object is masked from ‘package:datasets’:

    gait


Loading required package: Matrix

Funded by the Horizon 2020. Framework Programme of the European Union.



Attaching package: ‘dplyr’


The following object is masked from ‘package:MASS’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.2.0
✔ ggplot2   4.0.3     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.2     ✔ 

**Data**

In [3]:
data <- read_excel("DATA_LN3.xlsx")
head(data)

No,Tanggal,Waktu,Y,Hari,LN_NS,LN_M,LN_NM,NEWLN
<dbl>,<dttm>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,2020-01-01,00.30,3602.309,Rabu,1,0,0,1
2,2020-01-01,01.00,3525.583,Rabu,1,0,0,1
3,2020-01-01,01.30,3465.264,Rabu,1,0,0,1
4,2020-01-01,02.00,3400.974,Rabu,1,0,0,1
5,2020-01-01,02.30,3368.715,Rabu,1,0,0,1
6,2020-01-01,03.00,3326.610,Rabu,1,0,0,1


**Modeling**

In [6]:
Y <- data$Y[721:87648]
Ylag1 <- data$Y[673:87600]
Ylag2 <- data$Y[625:87552]
Ylag3 <- data$Y[577:87504]
Ylag4 <- data$Y[529:87456]
Ylag5 <- data$Y[481:87408]
Ylag6 <- data$Y[433:87360]
Ylag7 <- data$Y[385:87312]
Ylag8 <- data$Y[337:87264]
Ylag9 <- data$Y[289:87216]
Ylag10 <- data$Y[241:87168]
Ylag11 <- data$Y[193:87120]
Ylag12 <- data$Y[145:87072]
Ylag13 <- data$Y[97:87024]
Ylag14 <- data$Y[49:86976]
Ylag15 <- data$Y[1:86928]

#Dummy LN
LN <- data$NEWLN[721:87648]

#Dummy LN 3 type
NS <- data$LN_NS[721:87648]
M <- data$LN_M[721:87648]
NM <- data$LN_NM[721:87648]

n_day <- 1811
intervals_per_day <- 48
n <- n_day * intervals_per_day

# Slot waktu dalam sehari (1 sampai 48), diulang untuk 30 hari
slot <- rep(1:intervals_per_day, n_day)
slot_scaled <- (slot - 1) / (intervals_per_day - 1)

# --- 3 spesifikasi dummy ---
formulas <- list(
  TanpaDummy = Y ~ 1 + Ylag1 + Ylag7 + Ylag8 + Ylag14 + Ylag15,
  DummyLN    = Y ~ 1 + Ylag1 + Ylag7 + Ylag8 + Ylag14 + Ylag15 + LN,
  Dummy3Type = Y ~ 1 + Ylag1 + Ylag7 + Ylag8 + Ylag14 + Ylag15 + NS + M + NM
)

bws <- c(0.043, 0.086, 0.128)
ker <- "Epa"

# --- Grid 9 model. bw bervariasi paling cepat di dalam tiap spesifikasi dummy ---
configs <- expand.grid(bw = bws, spec = names(formulas),
                       KEEP.OUT.ATTRS = FALSE, stringsAsFactors = FALSE)

# --- Grid 48 titik intrahari ---
n_points    <- 48
slot_id     <- 1:n_points
time_scaled <- (slot_id - 1) / (n_points - 1)

# --- Wadah hasil ---
results <- vector("list", nrow(configs))
models  <- vector("list", nrow(configs))

# --- Loop 9 model: fit -> simpan 48 titik -> summary ---
for (i in seq_len(nrow(configs))) {
  spec_i <- configs$spec[i]
  bw_i   <- configs$bw[i]
  cat("\n>>> Menjalankan Model", i, "dari", nrow(configs),
      "|", spec_i, "| bw =", bw_i, "...\n"); flush.console()

  model_i <- tvLM(formulas[[spec_i]],
                  z = slot_scaled, bw = bw_i, tkernel = ker, est = "ll")

  coef_mat <- model_i$coefficients[1:n_points, , drop = FALSE]   # 48 slot
  results[[i]] <- data.frame(
    Slot        = slot_id,
    Time_scaled = time_scaled,
    coef_mat,
    check.names = FALSE
  )
  models[[i]] <- model_i
  names(results)[i] <- paste0("M", i, "_", spec_i, "_bw", bw_i)

  cat("\n===== Summary Model", i, "|", spec_i, "| bw =", bw_i, "=====\n")
  print(summary(model_i)); flush.console()
}

# --- Ekspor 9 sheet ---
write_xlsx(results, path = "Estimasi_FRTVCM_Dummy9Model.xlsx")


>>> Menjalankan Model 1 dari 9 | TanpaDummy | bw = 0.043 ...

===== Summary Model 1 | TanpaDummy | bw = 0.043 =====

Call: 
tvLM(formula = formulas[[spec_i]], z = slot_scaled, bw = bw_i, 
    est = "ll", tkernel = ker)

Class:  tvlm 

Summary of time-varying estimated coefficients: 
        (Intercept)  Ylag1  Ylag7   Ylag8 Ylag14  Ylag15
Min.          166.1 0.6431 0.4168 -0.4125 0.3781 -0.2903
1st Qu.       207.8 0.6568 0.4262 -0.3957 0.3850 -0.2720
Median        278.1 0.7550 0.4371 -0.3792 0.3900 -0.2597
Mean          284.6 0.7401 0.4452 -0.3787 0.3903 -0.2530
3rd Qu.       369.1 0.8081 0.4657 -0.3583 0.3958 -0.2300
Max.          400.3 0.8415 0.4718 -0.3434 0.4021 -0.2179

Bandwidth:  0.043
Pseudo R-squared:  0.8683 


Class:  tvlm 

Mean of coefficient estimates: 
(Intercept)       Ylag1       Ylag7       Ylag8      Ylag14      Ylag15 
   284.5639      0.7401      0.4452     -0.3787      0.3903     -0.2530 

Bandwidth:  0.043 


>>> Menjalankan Model 2 dari 9 | TanpaDummy | bw = 0.

In [7]:
# slot + tanggal
residual_df <- data.frame(
  Slot = slot,
  Tanggal = data$Tanggal[721:87648])

# tambah residual 9 model
for(i in seq_along(models)){
  residual_df[[paste0("Residual_M", i)]] <-
    models[[i]]$residuals}

# export 1 sheet
write_xlsx(
  list(Residual = residual_df),
  path = "Residual_9Model.xlsx")

**Diagnostic**

**Out of Sampel Accuracy**